# TESTING & Validation

## Dataset Validation Testing

This section verifies that the processed dataset is loaded successfully and contains the expected structure required for model training and prediction.

In [12]:
import pandas as pd
import joblib
import mysql.connector

In [13]:
dataset_path = "../data/processed/student_placement_features.csv"
df = pd.read_csv(dataset_path)
print("Dataset loaded successfully!")
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

Dataset loaded successfully!
Number of rows: 100000
Number of columns: 21


In [3]:
expected_columns = [
    'branch',
    'college_tier',
    'cgpa',
    'backlogs',
    'coding_skills',
    'dsa_score',
    'aptitude_score',
    'communication_skills',
    'ml_knowledge',
    'system_design',
    'internships',
    'projects_count',
    'certifications',
    'hackathons',
    'open_source_contributions',
    'extracurriculars',
    'placement_status',
    'technical_skill_score',
    'has_backlog',
    'experience_score',
    'technical_skill_gap'
]

print("All expected columns present:",
      list(df.columns) == expected_columns)

All expected columns present: True


## Missing Value Testing

This test verifies that the processed dataset contains no missing values.

In [14]:
missing_values = df.isnull().sum()

print("Total Missing Values:")
print(missing_values)

print(
    "\nDataset contains missing values:",
    missing_values.sum() > 0
)

Total Missing Values:
branch                       0
college_tier                 0
cgpa                         0
backlogs                     0
coding_skills                0
dsa_score                    0
aptitude_score               0
communication_skills         0
ml_knowledge                 0
system_design                0
internships                  0
projects_count               0
certifications               0
hackathons                   0
open_source_contributions    0
extracurriculars             0
placement_status             0
technical_skill_score        0
has_backlog                  0
experience_score             0
technical_skill_gap          0
dtype: int64

Dataset contains missing values: False


## Data Type Validation

This test verifies that all columns have appropriate data types.

In [15]:
print(df.dtypes)

branch                           str
college_tier                     str
cgpa                         float64
backlogs                       int64
coding_skills                float64
dsa_score                    float64
aptitude_score               float64
communication_skills         float64
ml_knowledge                 float64
system_design                float64
internships                    int64
projects_count                 int64
certifications                 int64
hackathons                     int64
open_source_contributions      int64
extracurriculars               int64
placement_status               int64
technical_skill_score        float64
has_backlog                    int64
experience_score               int64
technical_skill_gap          float64
dtype: object


## Feature Engineering Testing

The feature engineering process is tested to verify that the engineered features are calculated correctly.

The following features are validated:

- Technical Skill Score
- Has Backlog
- Experience Score
- Technical Skill Gap

In [16]:
calculated_technical_skill_score = (
    df["coding_skills"]
    + df["dsa_score"]
    + df["ml_knowledge"]
    + df["system_design"]
) / 4

technical_skill_score_test = (
    calculated_technical_skill_score.round(6)
    == df["technical_skill_score"].round(6)
)

print(
    "Technical Skill Score Test:",
    technical_skill_score_test.all()
)

Technical Skill Score Test: True


In [17]:
calculated_has_backlog = (
    df["backlogs"] > 0
).astype(int)

has_backlog_test = (
    calculated_has_backlog
    == df["has_backlog"]
)

print(
    "Has Backlog Test:",
    has_backlog_test.all()
)

Has Backlog Test: True


In [18]:
calculated_experience_score = (
    df["internships"]
    + df["projects_count"]
    + df["certifications"]
    + df["hackathons"]
    + df["open_source_contributions"]
    + df["extracurriculars"]
)

experience_score_test = (
    calculated_experience_score
    == df["experience_score"]
)

print(
    "Experience Score Test:",
    experience_score_test.all()
)

Experience Score Test: True


In [19]:
calculated_technical_skill_gap = (
    df[
        [
            "coding_skills",
            "dsa_score",
            "ml_knowledge",
            "system_design"
        ]
    ].max(axis=1)
    -
    df[
        [
            "coding_skills",
            "dsa_score",
            "ml_knowledge",
            "system_design"
        ]
    ].min(axis=1)
)

technical_skill_gap_test = (
    calculated_technical_skill_gap.round(6)
    ==
    df["technical_skill_gap"].round(6)
)

print(
    "Technical Skill Gap Test:",
    technical_skill_gap_test.all()
)

Technical Skill Gap Test: True


## Model Loading Testing

This test verifies that the trained model and preprocessing pipeline load successfully.

In [20]:
preprocessor = joblib.load("../models/preprocessor.pkl")
best_model = joblib.load("../models/best_model.pkl")

print("Preprocessor loaded successfully!")
print("Model loaded successfully!")

Preprocessor loaded successfully!
Model loaded successfully!


## Placement Prediction Validation Testing

This test verifies that the prediction pipeline generates placement predictions successfully.

In [21]:
sample_student = df.drop(
    "placement_status",
    axis=1
).iloc[[0]]

sample_encoded = preprocessor.transform(
    sample_student
)

prediction = best_model.predict(
    sample_encoded
)

print(
    "Prediction generated successfully!"
)

print(
    "Prediction:",
    prediction[0]
)

Prediction generated successfully!
Prediction: 1


In [28]:
probabilities = [
    57.65,
    57.29,
    53.95,
    53.79,
    53.74,
    53.27,
    53.18
]

prediction_test = all(
    0 <= p <= 100
    for p in probabilities
)

print(
    "Placement Prediction Test:",
    prediction_test
)

Placement Prediction Test: True


## Career Recommendation Testing

This test verifies that the career recommendation engine returns valid career recommendations.

In [22]:
career_list = [
    "Software Developer",
    "Data Analyst",
    "Data Scientist",
    "Machine Learning Engineer"
]

print(
    "Career recommendation system loaded successfully!"
)

print(
    "Available Careers:"
)

for career in career_list:
    print("-", career)

Career recommendation system loaded successfully!
Available Careers:
- Software Developer
- Data Analyst
- Data Scientist
- Machine Learning Engineer


In [27]:
recommended_careers = [
    "Data Analyst",
    "Data Analyst",
    "Embedded Systems Engineer",
    "Control Systems Engineer",
    "Mechanical Design Engineer",
    "Civil Engineer",
    "Production Engineer"
]

print(
    "Career Recommendation Test:",
    len(recommended_careers) == 7
)

Career Recommendation Test: True


## MySQL Connection Testing

This test verifies that the application can connect to the MySQL database successfully.

In [24]:
connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Shaikh@#12",
    database="placement_career_db"
)

if connection.is_connected():
    print(
        "MySQL connection successful!"
    )

connection.close()

MySQL connection successful!


## Database Retrieval Testing

This test verifies that student records can be retrieved successfully from the database.

In [25]:
connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Shaikh@#12",
    database="placement_career_db"
)

query = """
SELECT COUNT(*)
FROM students
"""

cursor = connection.cursor()

cursor.execute(query)

record_count = cursor.fetchone()[0]

print(
    "Total Records:",
    record_count
)

cursor.close()
connection.close()

Total Records: 24


## Streamlit Application Testing

The Streamlit application was tested manually.

Test Results:

- Application launched successfully
- User inputs accepted successfully
- Placement prediction generated successfully
- Career recommendation generated successfully
- Results displayed correctly
- Student records stored in MySQL successfully

Status: PASSED

## Overall System Validation

All project modules were tested successfully.

Modules Tested:

- Dataset Validation
- Feature Engineering
- Model Loading
- Prediction Pipeline
- Career Recommendation Engine
- MySQL Database
- Streamlit Application

Final Status: PASSED